# E-commerce Data Exploration

This notebook introduces a repeatable exploration workflow: load data with a deliberate schema, understand its structure, measure missing and invalid values, inspect distributions, and create useful summaries. The goal is to investigate before cleaning. A suspicious row is not automatically wrong—for example, negative quantities may represent returns and invoice numbers beginning with `C` may represent cancellations.

In [ ]:
from pathlib import Path

import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda value: f"{value:,.2f}")

DATA_PATH = Path(r"C:\data\ecomm\data.csv")
if not DATA_PATH.exists():
    raise FileNotFoundError(f"Input file not found: {DATA_PATH}")

## Type inference versus an explicit schema

By default, pandas samples values and infers a type for each column. This is convenient, but identifiers can be damaged: `CustomerID` becomes a float when blanks exist, and codes may lose leading zeros if interpreted as numbers. Dates initially remain text unless parsed. First inspect pandas' inference on a small sample; then load the complete file with intentional types.

In [ ]:
inferred_sample = pd.read_csv(DATA_PATH, encoding="latin1", nrows=1_000)
inferred_sample.dtypes.to_frame("inferred dtype")

`InvoiceNo` and `StockCode` are labels, not quantities, so they use `string`. `CustomerID` is also an identifier, but nullable `Int64` preserves missing values without converting IDs to decimals. `Quantity` is integral, `UnitPrice` is decimal, and `InvoiceDate` is parsed as a datetime. `latin1` is required because this file contains bytes that are not valid UTF-8.

In [ ]:
schema = {
    "InvoiceNo": "string",
    "StockCode": "string",
    "Description": "string",
    "Quantity": "int64",
    "UnitPrice": "float64",
    "CustomerID": "Int64",  # capital I means nullable integer
    "Country": "string",
}

df = pd.read_csv(
    DATA_PATH,
    encoding="latin1",
    dtype=schema,
    parse_dates=["InvoiceDate"],
    date_format="%m/%d/%Y %H:%M",
)

print(f"Rows: {len(df):,} | Columns: {df.shape[1]}")
df.head()

## Structure and a random sample

`info()` reports column types, non-null counts, and memory usage. A random sample often exposes variation that `head()` misses because adjacent CSV rows commonly belong to the same invoice. Use a fixed `random_state` so students see the same sample each time.

In [ ]:
df.info(memory_usage="deep")
df.sample(5, random_state=42)

## Missing-value profile

Missing counts show scale; percentages make columns comparable. Missing `CustomerID` does not necessarily make a transaction useless—it can indicate a guest customer while revenue and product fields remain valid. Missing descriptions may still be recoverable from other rows with the same stock code. Decide treatment according to the analysis, not with a blanket `dropna()`.

In [ ]:
quality_profile = pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "rows": len(df),
    "non_null": df.notna().sum(),
    "null_count": df.isna().sum(),
    "null_pct": df.isna().mean().mul(100),
    "unique_non_null": df.nunique(dropna=True),
})
quality_profile.sort_values("null_pct", ascending=False)

In [ ]:
# Do missing customer IDs occur more often in particular countries?
missing_customer_by_country = (
    df.assign(CustomerID_missing=df["CustomerID"].isna())
      .groupby("Country", observed=True)["CustomerID_missing"]
      .agg(missing_count="sum", missing_pct="mean", rows="size")
      .assign(missing_pct=lambda x: x["missing_pct"] * 100)
      .sort_values("missing_count", ascending=False)
)
missing_customer_by_country.head(10)

## Duplicates and business-rule checks

An exact duplicate repeats every field, but it may be either an ingestion duplicate or two identical purchased lines. Business rules locate values needing interpretation: cancellations, non-positive quantities or prices, blank text, and timestamps that failed parsing. Keep each flag separate so its cause remains visible and inspect examples before deciding what to exclude.

In [ ]:
flags = pd.DataFrame(index=df.index)
flags["exact_duplicate"] = df.duplicated(keep=False)
flags["cancelled_invoice"] = df["InvoiceNo"].str.startswith("C", na=False)
flags["quantity_non_positive"] = df["Quantity"] <= 0
flags["price_non_positive"] = df["UnitPrice"] <= 0
flags["customer_missing"] = df["CustomerID"].isna()
flags["description_missing_or_blank"] = (
    df["Description"].isna() | df["Description"].str.strip().eq("")
)
flags["date_invalid"] = df["InvoiceDate"].isna()

flag_summary = pd.DataFrame({
    "affected_rows": flags.sum(),
    "affected_pct": flags.mean().mul(100),
}).sort_values("affected_rows", ascending=False)
flag_summary

In [ ]:
# Inspect suspicious rows instead of immediately deleting them.
suspicious = flags.any(axis=1)
df.loc[suspicious].assign(**flags.loc[suspicious]).sample(10, random_state=42)

In [ ]:
# Check whether negative quantities align with cancellation invoice numbers.
pd.crosstab(
    flags["quantity_non_positive"],
    flags["cancelled_invoice"],
    rownames=["Quantity <= 0"],
    colnames=["Invoice starts with C"],
    margins=True,
)

## Numeric distributions and outliers

Summary statistics reveal range, center, and skew. Quantiles are especially useful when a few extreme orders make the mean misleading. The IQR rule below flags observations far outside the middle 50% of positive quantities. It is a screening method, not proof of an error: wholesale purchases can be both extreme and legitimate.

In [ ]:
df[["Quantity", "UnitPrice"]].describe(
    percentiles=[0.01, 0.05, 0.25, 0.50, 0.75, 0.95, 0.99]
).T

In [ ]:
positive_qty = df.loc[df["Quantity"] > 0, "Quantity"]
q1, q3 = positive_qty.quantile([0.25, 0.75])
iqr = q3 - q1
upper_fence = q3 + 1.5 * iqr
quantity_outliers = df[df["Quantity"] > upper_fence]

print(f"IQR upper fence: {upper_fence:,.2f}")
print(f"Rows above fence: {len(quantity_outliers):,}")
quantity_outliers.nlargest(10, "Quantity")

## Categorical exploration

Frequency tables reveal dominant groups, rare categories, spelling inconsistencies, and unexpectedly high cardinality. `nunique()` measures distinct values, while `value_counts()` shows their distribution. For free-text descriptions, normalize surrounding whitespace only in a copy; changing the raw data too early can hide original quality problems.

In [ ]:
for column in ["InvoiceNo", "StockCode", "Description", "CustomerID", "Country"]:
    print(f"{column}: {df[column].nunique(dropna=True):,} distinct non-null values")

df["Country"].value_counts(dropna=False).head(15).to_frame("rows")

## Create an analysis-ready sales view

Cleaning should be purpose-specific. For gross-sales exploration, remove exact duplicate rows, exclude cancellation records, and require positive quantity and price. Keep missing customer IDs because anonymous purchases still contribute revenue. Fill missing descriptions only for display and retain the original `df` unchanged so returns and quality issues remain available for separate analysis.

In [ ]:
sales = df.drop_duplicates().copy()
sales["Description"] = sales["Description"].str.strip().fillna("Unknown product")
sales["Country"] = sales["Country"].str.strip()
sales = sales.loc[
    ~sales["InvoiceNo"].str.startswith("C", na=False)
    & sales["Quantity"].gt(0)
    & sales["UnitPrice"].gt(0)
].copy()

sales["LineRevenue"] = sales["Quantity"] * sales["UnitPrice"]
sales["InvoiceMonth"] = sales["InvoiceDate"].dt.to_period("M")
sales["InvoiceHour"] = sales["InvoiceDate"].dt.hour
sales["DayName"] = sales["InvoiceDate"].dt.day_name()

print(f"Raw rows: {len(df):,} | Analysis-ready sales rows: {len(sales):,}")
sales.head()

## Sales summaries

Aggregation changes transaction lines into business measures. Revenue is quantity multiplied by unit price. Grouping by month exposes trends; grouping by country or product supports comparisons. Always show context such as line count, invoice count, or units beside revenue—otherwise a large total may be wrongly interpreted as many customers rather than a few large orders.

In [ ]:
monthly_sales = (
    sales.groupby("InvoiceMonth", observed=True)
         .agg(
             revenue=("LineRevenue", "sum"),
             units=("Quantity", "sum"),
             invoices=("InvoiceNo", "nunique"),
             customers=("CustomerID", "nunique"),
         )
)
monthly_sales

In [ ]:
country_sales = (
    sales.groupby("Country", observed=True)
         .agg(revenue=("LineRevenue", "sum"),
              invoices=("InvoiceNo", "nunique"),
              customers=("CustomerID", "nunique"))
         .sort_values("revenue", ascending=False)
)
country_sales.head(10)

In [ ]:
top_products = (
    sales.groupby(["StockCode", "Description"], observed=True)
         .agg(units=("Quantity", "sum"),
              revenue=("LineRevenue", "sum"),
              invoices=("InvoiceNo", "nunique"))
         .sort_values("revenue", ascending=False)
)
top_products.head(10)

## Invoice-level exploration

The CSV grain is one product line, not one invoice. To answer basket questions, aggregate lines by invoice first. This avoids counting a multi-product invoice as several separate orders. Invoice-level metrics include basket revenue, total units, and distinct products; their median is often more representative than the mean when large wholesale orders exist.

In [ ]:
invoice_summary = (
    sales.groupby("InvoiceNo", observed=True)
         .agg(
             invoice_date=("InvoiceDate", "min"),
             customer_id=("CustomerID", "first"),
             country=("Country", "first"),
             basket_revenue=("LineRevenue", "sum"),
             units=("Quantity", "sum"),
             distinct_products=("StockCode", "nunique"),
         )
)
invoice_summary[["basket_revenue", "units", "distinct_products"]].describe().T

## Lightweight validation

Assertions turn assumptions into executable checks. These checks do not claim the source is perfect; they confirm that the sales view obeys the rules chosen above. If future data violates a rule, execution stops close to the cause instead of allowing misleading summaries to continue. Production pipelines usually replace bare assertions with logged validation reports or dedicated quality tools.

In [ ]:
expected_columns = {
    "InvoiceNo", "StockCode", "Description", "Quantity",
    "InvoiceDate", "UnitPrice", "CustomerID", "Country",
}
assert expected_columns == set(df.columns), "Unexpected or missing input columns"
assert sales["InvoiceNo"].notna().all(), "Sales invoice numbers must be present"
assert sales["Quantity"].gt(0).all(), "Sales quantities must be positive"
assert sales["UnitPrice"].gt(0).all(), "Sales prices must be positive"
assert sales["InvoiceDate"].notna().all(), "Sales dates must parse successfully"
assert sales["LineRevenue"].gt(0).all(), "Sales line revenue must be positive"
print("All analysis-view checks passed.")

## Further questions

Try comparing return rates by product, country, and month; identifying customers by recency, frequency, and monetary value (RFM); checking whether one stock code has multiple descriptions; plotting hourly or weekday demand; and studying cohort retention. Before each analysis, state the row grain, define valid records, and document whether returns, cancellations, anonymous customers, and outliers are included.

## Add a derived amount column

`Amount` is the value of each transaction line: quantity multiplied by unit price. It is added to the original DataFrame so future features can build on it. Negative quantities produce negative amounts, preserving returns or cancellations for later analysis.

In [ ]:
df["Amount"] = df["Quantity"] * df["UnitPrice"]


In [ ]:

countries = sorted(df["Country"].dropna().unique().tolist())
print(f"Unique countries: {len(countries)}")
print(countries)

country_summary = (
    df.groupby("Country", observed=True)
      .agg(
          InvoiceCount=("InvoiceNo", "nunique"),
          TotalAmount=("Amount", "sum"),
      )
      .sort_values("TotalAmount", ascending=False)
)
country_summary

## Most valued customers

Customer value is ranked by total amount spent on positive, non-cancelled purchases. Rows without a customer ID cannot be assigned to an individual and are excluded. Invoice count and units provide context: two customers can spend the same amount through very different buying patterns. `rank(method="dense")` gives tied customers the same rank without gaps.

In [ ]:
customer_purchases = df.loc[
    df["CustomerID"].notna()
    & ~df["InvoiceNo"].str.startswith("C", na=False)
    & df["Quantity"].gt(0)
    & df["UnitPrice"].gt(0)
].copy()

customer_ranking = (
    customer_purchases.groupby("CustomerID", observed=True)
      .agg(
          TotalSpent=("Amount", "sum"),
          InvoiceCount=("InvoiceNo", "nunique"),
          UnitsPurchased=("Quantity", "sum"),
          LastPurchase=("InvoiceDate", "max"),
      )
      .sort_values("TotalSpent", ascending=False)
)
customer_ranking["AverageInvoiceValue"] = (
    customer_ranking["TotalSpent"] / customer_ranking["InvoiceCount"]
)
customer_ranking["Rank"] = (
    customer_ranking["TotalSpent"].rank(method="dense", ascending=False).astype("int64")
)
customer_ranking = customer_ranking.reset_index().set_index("Rank")

customer_ranking.head(20)

## Most sold products by SKU

Products are ranked by total units sold, rather than row count. Only positive, non-cancelled sales with a positive price are included, but anonymous-customer purchases remain valid. Revenue and distinct invoice count provide context. A stock code can have multiple descriptions, so the most frequently occurring non-null description is selected as its representative label.

In [ ]:
valid_product_sales = df.loc[
    ~df["InvoiceNo"].str.startswith("C", na=False)
    & df["Quantity"].gt(0)
    & df["UnitPrice"].gt(0)
].copy()

sku_descriptions = (
    valid_product_sales.dropna(subset=["Description"])
      .groupby("StockCode", observed=True)["Description"]
      .agg(lambda values: values.value_counts().index[0])
)

sku_ranking = (
    valid_product_sales.groupby("StockCode", observed=True)
      .agg(
          UnitsSold=("Quantity", "sum"),
          TotalRevenue=("Amount", "sum"),
          InvoiceCount=("InvoiceNo", "nunique"),
      )
      .join(sku_descriptions.rename("Description"))
      .sort_values(["UnitsSold", "TotalRevenue"], ascending=False)
)
sku_ranking["Rank"] = (
    sku_ranking["UnitsSold"].rank(method="dense", ascending=False).astype("int64")
)
sku_ranking = sku_ranking.reset_index().set_index("Rank")

sku_ranking.head(20)

## Invoice-level purchase ranking

An invoice can contain many product lines, so ranking individual rows does not rank complete orders. Here, `TotalAmount` and `TotalQuantity` are summed per invoice. `TotalItems` counts rows—the number of item entries—irrespective of the quantity recorded on each row. Positive, non-cancelled sales are used so the ranking represents purchases rather than returns.

In [ ]:
invoice_ranking = (
    valid_product_sales.groupby("InvoiceNo", observed=True)
      .agg(
          TotalAmount=("Amount", "sum"),
          TotalQuantity=("Quantity", "sum"),
          TotalItems=("StockCode", "size"),  # row/entry count
      )
      .sort_values("TotalAmount", ascending=False)
)
invoice_ranking["Rank"] = (
    invoice_ranking["TotalAmount"].rank(method="dense", ascending=False).astype("int64")
)
invoice_ranking = invoice_ranking.reset_index().set_index("Rank")

top_100_invoices = invoice_ranking.head(100)
bottom_100_invoices = invoice_ranking.tail(100).sort_values("TotalAmount")

print("Top 100 invoices by total amount")
display(top_100_invoices)

print("Bottom 100 invoices by total amount")
display(bottom_100_invoices)

## Negative quantities and estimated loss

Negative quantities indicate items leaving recorded sales or inventory. An invoice beginning with `C` strongly suggests a cancellation or return; other rows may be stock adjustments, damage, missing goods, or corrections. Because the dataset has no cost or reason field, true profit loss cannot be calculated. `EstimatedLoss` below is the absolute recorded sales value reversed, while zero-priced adjustments report lost units but no monetary value.

In [ ]:
negative_qty = df.loc[df["Quantity"].lt(0)].copy()
negative_qty["ReasonGroup"] = (
    negative_qty["InvoiceNo"].str.startswith("C", na=False)
      .map({True: "Cancellation / return", False: "Stock or administrative adjustment"})
)
negative_qty["UnitsRemoved"] = -negative_qty["Quantity"]
negative_qty["EstimatedLoss"] = (-negative_qty["Amount"]).clip(lower=0)

negative_quantity_totals = pd.Series({
    "NegativeQuantityRows": len(negative_qty),
    "UnitsRemoved": negative_qty["UnitsRemoved"].sum(),
    "RowsMissingCustomerID": negative_qty["CustomerID"].isna().sum(),
    "EstimatedGrossValueLoss": negative_qty["EstimatedLoss"].sum(),
})
negative_quantity_totals

In [ ]:
negative_reason_summary = (
    negative_qty.groupby("ReasonGroup", observed=True)
      .agg(
          Rows=("InvoiceNo", "size"),
          UnitsRemoved=("UnitsRemoved", "sum"),
          MissingCustomerRows=("CustomerID", lambda values: values.isna().sum()),
          EstimatedLoss=("EstimatedLoss", "sum"),
      )
      .sort_values("EstimatedLoss", ascending=False)
)
negative_reason_summary

Descriptions can provide clues for non-cancellation adjustments, but they are unstructured text rather than reliable reason codes. Review their frequencies before creating categories. A zero estimated loss means the source recorded a zero unit price; it does not mean damaged or missing inventory had no real cost.

In [ ]:
adjustment_descriptions = (
    negative_qty.loc[
        negative_qty["ReasonGroup"].eq("Stock or administrative adjustment"),
        "Description",
    ]
    .fillna("Missing description")
    .value_counts()
    .head(20)
)
adjustment_descriptions.to_frame("Rows")

In [ ]:
# Products with the greatest recorded value reversed.
negative_product_loss = (
    negative_qty.groupby(["StockCode", "Description"], dropna=False, observed=True)
      .agg(
          UnitsRemoved=("UnitsRemoved", "sum"),
          AffectedRows=("InvoiceNo", "size"),
          EstimatedLoss=("EstimatedLoss", "sum"),
      )
      .sort_values("EstimatedLoss", ascending=False)
)
negative_product_loss.head(20)